# TCG-pokebot — deck-AGNOSTIC training v2.1 (single Save-&-Run-All)

One net that pilots the **whole field**, so self-play is honest. Built to run end-to-end on a 12h Kaggle kernel: it **checkpoints every iteration**, **stops before the kill**, **resumes across sessions**, and **persists weights** to a dataset.

## Add in the right-hand panel before Run-All
1. **Competitions** → the *Pokémon TCG AI Battle* competition (engine `cg/`, `EN_Card_Data.csv`). Must have **joined**.
2. **Datasets** (resume, optional) → your `<username>/tcg-pokebot-weights` if a previous run made one.
3. **Code** → public repo (Internet ON), or a `GITHUB_TOKEN` secret for a private repo, or set `CODE_DATASET`.
4. **Secrets** (Add-ons → Secrets) → `KAGGLE_USERNAME` + `KAGGLE_KEY` so weights persist to a dataset (else they stay in `/kaggle/working`).

This is the agnostic build; a deck-specialist net is a later, separate step.

## 1. Config

In [ ]:
USERNAME        = 'hakase0'
REPO            = 'Hakase-0/TCG-pokebot'
CODE_DATASET    = ''                               # fallback if offline: '<user>/tcg-pokebot-code'
WEIGHTS_DATASET = f'{USERNAME}/tcg-pokebot-weights'

# --- net capacity (COLD-START upgrade; a net-size change auto-cold-starts -- see cell 6) ---
# 192/6/3 (~2.0M) -> 256/4/3 (~3.0M) on this fresh start: a wider trunk for the harder
# agnostic task, and 4 heads => head-dim 256/4=64 (the 64-128 sweet spot) vs the old 192/6=32.
# Depth stays at 3 LAYERS on purpose -- self-play inference runs on CPU workers regardless of
# the accelerator, and extra layers add sequential latency (wide matmuls vectorize better than
# deep ones), so this buys capacity for the least self-play throughput. WATCH games/iter in
# logs/rl.jsonl vs the old run; if it drops >~25%, fall back to 192/3/3 (head-dim 64, ~free).
D, N_HEADS, N_LAYERS = 256, 4, 3

# --- compute target ---
DEVICE          = 'auto'      # 'auto' = detect at runtime (cell 7) what this kernel actually got, trying
                              #   usable CUDA GPU -> TPU -> CPU in that order. Override with 'gpu', 'tpu',
                              #   or 'cpu' to force one. To GET a TPU you must also select 'TPU VM v3-8' in
                              #   the editor Settings (or set "machine_shape":"Tpu1VmV38" for a CLI push);
                              #   TPU VMs ship PyTorch/XLA preinstalled. When TPU is chosen, TCG_DEVICE=tpu
                              #   is scoped to the RL step so ONLY the batched train step parks on the TPU;
                              #   self-play + inference stay on CPU (XLA recompiles on the variable option
                              #   width). Switch the accelerator on the website and re-run; 'auto' adapts.

# --- training ---
PHASE           = 'value'     # ISMCTS leaf: 'value' = net value head (fast; earned via the d=192 A/B and
                              #   sharpened + carried forward each run by the SHARPEN cell) | 'rollout' =
                              #   engine-oracle heuristic (slow; only for a FRESH cold start, before the
                              #   value head has been trained by an RL run).
OPP_SEARCH      = 'raw'       # opponent move: 'raw' net argmax (fast, default) | 'ismcts' symmetric search-vs-search.
                              #   Switch to 'ismcts' once you switch PHASE='value' (search becomes cheap); ~2x cost on rollout.
BC_GAMES        = 150         # agnostic BC games (combat teacher pilots the whole pool); first run only
BC_EPOCHS       = 15
ITERS           = 8           # cap raised so the TIME BUDGET governs, not the cap
GAMES           = 60          # self-play games / iter (kept large for a robust per-iter signal; ~160min/iter at sims=48)
ISMCTS_WORLDS, ISMCTS_SIMS = 3, 48   # stronger search than 2,24: sharper policy targets + a better value-head bootstrap.
                                     #   ~3x self-play cost; the field eval pilots the RAW net, so it's unaffected.
LEAGUE_EVERY    = 2           # gateless (AlphaZero-style): snapshot the candidate into the self-play league every
                              #   N iters for opponent diversity. Replaces the old promotion gate; costs no arena
                              #   games (a deepcopy). 0 disables league growth.
FIELD_GAMES     = 12          # x3 internally for the agnostic field metric
TIME_BUDGET_MIN = 420         # Kaggle kills at 720; checked BETWEEN iters, so leave room for one more
                              #   in-flight iter + value-head sharpen (minutes) + persist before the kill.
                              #   The old ~70min A/B is gone and the 'value' leaf iterates faster than
                              #   'rollout', so there's headroom to raise this if you want more iters.

WORKERS         = 0           # parallel self-play worker PROCESSES (0 = auto = os.cpu_count()). Self-play is
                              #   CPU/engine-bound (engine search + batch-1 net eval), so this multiplies
                              #   games/iter ~linearly with cores; workers do CPU inference, the GPU is reserved
                              #   for the training step. 1 = original single-process loop. Falls back to
                              #   sequential on any multiprocessing error.

import os, sys, glob, shutil, subprocess, time
WORKDIR='/kaggle/working/TCG-pokebot'
def banner(t): print('\n'+'='*70+f'\n  {t}\n'+'='*70, flush=True)
print('config OK | net', (D,N_HEADS,N_LAYERS), '| phase', PHASE, '| opp', OPP_SEARCH)

## 2. Code

In [ ]:
banner('SETUP: code')
def have_code(p): return os.path.exists(os.path.join(p,'ismcts.py')) and os.path.exists(os.path.join(p,'selfplay_rl.py'))
def _tok():
    try:
        from kaggle_secrets import UserSecretsClient; return UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception: return None
if not have_code(WORKDIR):
    done=False; tok=_tok()
    for url in [u for u in (f'https://{tok}@github.com/{REPO}' if tok else None, f'https://github.com/{REPO}') if u]:
        try:
            subprocess.run(['git','clone','--depth','1',url,WORKDIR],check=True); done=have_code(WORKDIR)
            if done: print('cloned from GitHub'); break
        except Exception as e: print('clone failed:', str(e)[:100])
    if not done and CODE_DATASET:
        src=f'/kaggle/input/{CODE_DATASET.split("/")[-1]}'
        cand=next((d for d,_,f in os.walk(src) if 'ismcts.py' in f),None)
        if cand:
            os.makedirs(WORKDIR,exist_ok=True)
            for f in glob.glob(os.path.join(cand,'*')):
                (shutil.copytree if os.path.isdir(f) else shutil.copy)(f,os.path.join(WORKDIR,os.path.basename(f)))
            done=have_code(WORKDIR); print('copied from code dataset:', done)
    assert done,'could not obtain code (see cell 0)'
else: print('code already present')
os.chdir(WORKDIR); sys.path.insert(0,WORKDIR)
print('cwd', os.getcwd(), '|', len(glob.glob('*.py')),'py files')


## 3. Engine + 4. Tables + 5. Deck pool (offline)

In [ ]:
banner('SETUP: engine, tables, deck pool')
# engine from the competition input
if not os.path.exists(os.path.join(WORKDIR,'cg','libcg.so')):
    so=next((os.path.join(d,'libcg.so') for d,_,fs in os.walk('/kaggle/input') if 'libcg.so' in fs),None)
    assert so,'libcg.so not found under /kaggle/input — add the competition'
    shutil.copytree(os.path.dirname(so),os.path.join(WORKDIR,'cg'),dirs_exist_ok=True); print('engine copied')
if not os.path.exists('EN_Card_Data.csv'):
    src=next((os.path.join(d,'EN_Card_Data.csv') for d,_,fs in os.walk('/kaggle/input') if 'EN_Card_Data.csv' in fs),None)
    if src: shutil.copy(src,'EN_Card_Data.csv')
os.environ['CG_LIB_PATH']=os.path.join(WORKDIR,'cg')
from cg import api; print('engine OK — cards', len(api.all_card_data()))
# tables
subprocess.run([sys.executable,'inspect_cards.py'],check=True,env={**os.environ})
print('tables:', os.path.exists('capability_table.json'))
# offline deck pool: txt -> id-csv (no internet)
from import_deck import import_decklist, write_deck_csv
# OPTIONAL online expansion (Kaggle has network): grow the off-meta bucket toward OFFMETA_TARGET
# decks. Off-meta now counts in the field metric (selfplay_rl field_pool = meta + adversary), so
# more rogue lists => better agnostic robustness. NON-DESTRUCTIVE + best-effort: fetched decks land
# under an 'advx-*' prefix so the committed adversary baseline is never touched, and ANY failure
# (offline, parse drift) just falls back to the committed decks. Meta stays at the committed top-30.
OFFMETA_TARGET = 20
_have_adv = len(glob.glob('decks/adversary/*.txt'))
if _have_adv < OFFMETA_TARGET and os.path.exists('EN_Card_Data.csv'):
    try:
        import build_deck_pool as BDP
        n_new = BDP.build_from_archetype('other', OFFMETA_TARGET - _have_adv,
                                         'EN_Card_Data.csv', 'decks/adversary', 'advx')
        print(f'off-meta expand: +{n_new} fetched (advx-*) on top of {_have_adv} committed')
    except Exception as e:
        print('off-meta expand skipped (committed adversary decks kept):', str(e)[:80])
for txt in sorted(glob.glob('decks/*.txt'))+sorted(glob.glob('decks/adversary/*.txt')):
    csv=txt[:-4]+'.csv'
    if not os.path.exists(csv):
        try:
            ids,_=import_decklist(open(txt).read(),'EN_Card_Data.csv')
            if len(ids)==60: write_deck_csv(ids,csv)
        except Exception as e: print('skip',os.path.basename(txt),str(e)[:60])
print('deck pool:', len(glob.glob('decks/*.csv')),'meta +', len(glob.glob('decks/adversary/*.csv')),'adversary')


In [ ]:
banner('ACCELERATOR: detect GPU / TPU / CPU and match torch to it')
# Figures out what THIS kernel actually has and adapts -- so you can flip the accelerator
# dropdown on the website and Run-All without touching code (DEVICE='auto'), or force one
# with DEVICE in {'gpu','tpu','cpu'}. Probes run in SUBPROCESSES: a bad CUDA kernel can't
# wedge this cell, and torch_xla never initializes in THIS process (self-play workers, which
# fork from here, would otherwise inherit a half-init XLA runtime).
#
# P100 gotcha: Kaggle's GPU may be a P100 (Pascal, sm_60). Recent torch wheels dropped sm_60,
# so reinstalling 'latest' would NOT fix a P100. Pin to 2.4.1+cu121, whose official wheel ships
# 5.0;6.0;7.0;7.5;8.0;8.6;9.0 -- covering BOTH P100 (sm_60) and the T4 (sm_75) Kaggle assigns.
TORCH_PIN = 'torch==2.4.1'   # last broadly-shipped wheel that still includes sm_60 (Pascal/P100)
_PROBE = r"""
import torch
if not torch.cuda.is_available():
    print('NOGPU')
else:
    print('DEV', torch.cuda.get_device_name(0), 'sm', torch.cuda.get_device_capability(0))
    torch.zeros(1, device='cuda').add_(1.0); torch.cuda.synchronize()
    print('USABLE', torch.__version__)
"""
def _gpu_probe():
    r = subprocess.run([sys.executable, '-c', _PROBE], capture_output=True, text=True)
    dev = next((l for l in r.stdout.splitlines() if l.startswith('DEV')), '')
    if 'USABLE' in r.stdout: return 'USABLE', (dev or r.stdout.strip())
    if 'NOGPU'  in r.stdout: return 'NOGPU',  'no CUDA device'
    return 'BROKEN', (dev + ' | ' + (r.stderr.strip().splitlines() or ['?'])[-1])[:160]
def _tpu_reachable():
    # torch_xla is preinstalled only on Kaggle TPU VMs; reaching a device confirms a real TPU.
    probe = "import torch_xla.core.xla_model as xm; print('TPUOK', xm.xla_device())"
    try:
        r = subprocess.run([sys.executable, '-c', probe], capture_output=True, text=True,
                           env={**os.environ, 'PJRT_DEVICE': 'TPU'}, timeout=180)
    except Exception:
        return ''
    return next((l.split(' ', 1)[1] for l in r.stdout.splitlines() if l.startswith('TPUOK')), '')
def detect_accelerator():
    if DEVICE in ('cpu', 'gpu', 'tpu'):
        print(f' DEVICE={DEVICE!r} (forced)'); return DEVICE
    # auto: usable CUDA GPU first (commonest), reinstalling a Pascal-safe wheel if the stock
    # one is broken; then a reachable TPU; else CPU.
    tag, msg = _gpu_probe(); print(' gpu probe:', tag, '|', msg)
    if tag == 'BROKEN':
        print(f'  stock torch cannot run on this GPU -> reinstalling {TORCH_PIN}+cu121 (Pascal-safe) ~2-4 min...', flush=True)
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', TORCH_PIN,
                            '--index-url', 'https://download.pytorch.org/whl/cu121',
                            '--extra-index-url', 'https://pypi.org/simple'],
                           capture_output=True, text=True)
        print('  pip rc', r.returncode, '|', (r.stderr.strip() or r.stdout.strip())[-200:])
        tag, msg = _gpu_probe(); print(' gpu re-probe:', tag, '|', msg)
    if tag == 'USABLE':
        return 'gpu'
    tpu = _tpu_reachable()
    if tpu:
        print(' tpu probe: reachable |', tpu); return 'tpu'
    print(' tpu probe: none'); return 'cpu'
ACCEL = detect_accelerator()
USE_TPU = (ACCEL == 'tpu')
print(' -> training will use', ACCEL.upper(),
      '(TCG_DEVICE=tpu for the RL step)' if USE_TPU else '(device() falls back automatically; no crash)')

## 6. Resume — newest checkpoint (if a weights dataset is attached)

In [ ]:
banner('RESUME')
def find_ckpt():
    roots=[f'/kaggle/input/{WEIGHTS_DATASET.split("/")[-1]}','/kaggle/working',WORKDIR]
    for pat in ('*.latest.pt','rl_model.pt','model.pt'):
        for r in roots:
            h=sorted(glob.glob(os.path.join(r,'**',pat),recursive=True),key=os.path.getmtime,reverse=True)
            if h: return h[0]
    return None
# also recover model_meta.json from a checkpoint dir so dims match on resume
WARM=find_ckpt()
if WARM:
    # COLD START on a net-size change: if the checkpoint's architecture differs from the
    # configured (D,N_HEADS,N_LAYERS) its weights cannot load -- so start a FRESH net instead
    # of crashing. The value-data recovery below still runs, so a cold start KEEPS the harvested
    # training data (replay buffer preload) and only discards the incompatible old weights.
    import json
    md_src=os.path.join(os.path.dirname(WARM),'model_meta.json')
    meta={}
    try: meta=json.load(open(md_src)) if os.path.exists(md_src) else {}
    except Exception as e: print('  (could not read checkpoint meta:',str(e)[:60],')')
    ckpt_arch=(meta.get('d'),meta.get('n_heads'),meta.get('n_layers'))
    if ckpt_arch!=(D,N_HEADS,N_LAYERS):
        print(f'  checkpoint arch {ckpt_arch} != config {(D,N_HEADS,N_LAYERS)} -> COLD START (fresh net; data kept)')
        WARM=None
    elif not os.path.exists('model_meta.json'):
        shutil.copy(md_src,'model_meta.json')   # arch matches -> reuse meta on resume
# recover prior-session value-data dumps so the replay buffer can PRELOAD past games
# (weights persist across sessions, but self-play decisions otherwise wouldn't — this
# carries the harvested games forward so training compounds instead of restarting cold).
src=f'/kaggle/input/{WEIGHTS_DATASET.split("/")[-1]}'
if os.path.isdir(src):
    os.makedirs('valuedata',exist_ok=True); n=0
    for p in glob.glob(os.path.join(src,'**','*.pkl'),recursive=True):
        b=os.path.basename(p)
        if 'iter' in b and not os.path.exists(os.path.join('valuedata',b)):
            shutil.copy(p,os.path.join('valuedata',b)); n+=1
    print('recovered',n,'prior value-data dumps ->',len(glob.glob('valuedata/*.pkl')),'total in valuedata/')
NEED_BC = WARM is None
print('resume from:', WARM if WARM else 'NONE -> agnostic BC bootstrap (cell 7)')


## 7. (first run only) Agnostic BC bootstrap
Combat teacher pilots **every deck in the pool**, distilled into the bigger net. Skipped if resuming.

In [ ]:
if NEED_BC:
    banner('BC BOOTSTRAP (agnostic, combat teacher over the field)')
    subprocess.run([sys.executable,'-u','gen_selfplay_data.py','--games',str(BC_GAMES),
                    '--policy','combat','--out','data/bc.pkl','--log','logs/sp.jsonl'],check=True,env={**os.environ})
    subprocess.run([sys.executable,'-u','train_bc.py','--data','data/bc.pkl','--epochs',str(BC_EPOCHS),
                    '--out','model.pt','--d',str(D),'--n-heads',str(N_HEADS),'--n-layers',str(N_LAYERS),
                    '--log','logs/tr.jsonl'],check=True,env={**os.environ})
    WARM='model.pt'; print('BC done ->', WARM)
else: print('skipping BC; resuming from', WARM)


## 8. Agnostic RL — time-budgeted, checkpointed, harvests value data

In [ ]:
banner(f'RL (agnostic, leaf={PHASE}, opp={OPP_SEARCH})')
OUT,LATEST='rl_model.pt','rl_model.latest.pt'
# Replay buffer instead of extra epochs: training several epochs on ONE iteration's ~4.5k
# highly self-correlated decisions overfits and degrades general strength (the iter-1
# field-eval drop, 47% vs 58% baseline, was exactly this). So keep --epochs at the default 1
# and pass --replay-buffer: each iter trains on a decorrelated sliding window of recent games
# (AlphaZero/KataGo-style), a more stable update on more data. Gateless (no promotion gate):
# rl_model.pt and rl_model.latest.pt both hold the latest trained net, written every iter.
# --replay-preload warms that buffer from value-data recovered in cell 6, so the window spans
# PAST sessions too (games persist across runs now, not just weights).
cmd=[sys.executable,'-u','selfplay_rl.py','--search','ismcts','--leaf',PHASE,
     '--opp-search',OPP_SEARCH,'--replay-buffer','30000','--replay-preload','valuedata','--workers',str(WORKERS),
     '--ismcts-worlds',str(ISMCTS_WORLDS),'--ismcts-sims',str(ISMCTS_SIMS),
     '--iters',str(ITERS),'--games',str(GAMES),'--league-every',str(LEAGUE_EVERY),
     '--field-games',str(FIELD_GAMES),'--time-budget-min',str(TIME_BUDGET_MIN),
     '--warm',WARM,'--opp-decks','decks/','--out',OUT,'--latest-out',LATEST,
     '--dump-samples','valuedata','--log','logs/rl.jsonl']
print(' '.join(cmd),flush=True)
# USE_TPU (resolved in cell 7) -> scope TCG_DEVICE to THIS subprocess so selfplay_rl.device()
# selects the TPU for the batched train step only; self-play workers never inherit it (CPU).
rl_env={**os.environ, **({'TCG_DEVICE':'tpu'} if USE_TPU else {})}
subprocess.run(cmd,check=True,env=rl_env)
print('RL segment done. checkpoints:', [f for f in (OUT,LATEST) if os.path.exists(f)])

## 9. Sharpen the value head (carried forward)
Minutes-long, frozen-body value-head fit on the harvested data (policy provably unchanged), then the sharpened head is copied onto `rl_model.pt`/`rl_model.latest.pt` so the **fast** `value` leaf uses it next run. The old value-vs-rollout A/B is removed — the switch to `PHASE='value'` is already decided.

In [ ]:
banner('VALUE-HEAD SHARPEN')
# PHASE='value': the net's value head IS the ISMCTS leaf now, so sharpening it is a direct
# search-quality win -- and it's policy-safe (fit_value freezes the body: 0.0 policy-logit drift).
# Carry the sharpened net forward onto the resume/output checkpoints so the NEXT run's value leaf
# uses it. The old value-vs-rollout A/B is removed: the switch is decided (value >= rollout within
# CI on the d=192 A/B), so re-checking it every session just burns ~80 arena games of compute.
BASE = next((p for p in ('rl_model.pt','rl_model.latest.pt') if os.path.exists(p)), '')
if glob.glob('valuedata/*.pkl') and BASE:
    print('value-head base:', BASE, flush=True)
    subprocess.run([sys.executable,'fit_value.py','--data','valuedata','--warm',BASE,
                    '--out','rl_model.value.pt','--epochs','8','--target-blend','0.5'],check=True,env={**os.environ})
    for dst in ('rl_model.pt','rl_model.latest.pt'):   # same policy, sharper value head ->
        shutil.copy('rl_model.value.pt',dst)           #   becomes the leaf + resume net next run
    print('sharpened value head -> rl_model.pt + rl_model.latest.pt (carried forward as the leaf)',flush=True)
else: print('no value data yet')

## 10. Persist weights (survives the notebook)

In [ ]:
banner('PERSIST WEIGHTS')
import json
STAGE='/kaggle/working/weights'; os.makedirs(STAGE,exist_ok=True)
for f in glob.glob('rl_model*.pt')+['model.pt','model_meta.json']+glob.glob('logs/*.jsonl'):
    if os.path.exists(f): shutil.copy(f,os.path.join(STAGE,os.path.basename(f)))
# also persist recent value-data dumps so NEXT session's replay buffer can preload them
# (cross-session game continuity: cell 6 recovers them, cell 8 preloads them). Session-unique
# filenames -> they upload as individual files; keep only the newest VKEEP so the dataset
# doesn't grow without bound.
VKEEP=12   # ~ last 4 sessions at 3 iters/session; the buffer itself only uses the newest ~30k decisions
for f in sorted(glob.glob('valuedata/*.pkl'))[-VKEEP:]:
    shutil.copy(f,os.path.join(STAGE,os.path.basename(f)))
print('saved to /kaggle/working/weights:', os.listdir(STAGE))
def _creds():
    try:
        from kaggle_secrets import UserSecretsClient; u=UserSecretsClient()
        os.environ['KAGGLE_USERNAME']=u.get_secret('KAGGLE_USERNAME'); os.environ['KAGGLE_KEY']=u.get_secret('KAGGLE_KEY'); return True
    except Exception as e: print('no KAGGLE_* secrets -> dataset push skipped (weights are in /kaggle/working):',str(e)[:70]); return False
if _creds():
    json.dump({'title':'tcg-pokebot-weights','id':WEIGHTS_DATASET,'licenses':[{'name':'CC0-1.0'}]},
              open(os.path.join(STAGE,'dataset-metadata.json'),'w'))
    msg=f'agnostic weights {time.strftime("%Y-%m-%d %H:%M")} phase={PHASE}'
    r=subprocess.run(['kaggle','datasets','version','-p',STAGE,'-m',msg,'--dir-mode','zip'],capture_output=True,text=True)
    if r.returncode!=0 and 'not found' in (r.stdout+r.stderr).lower():
        r=subprocess.run(['kaggle','datasets','create','-p',STAGE,'--dir-mode','zip'],capture_output=True,text=True)
    print(r.stdout[-300:] or r.stderr[-300:]); print('-> attach',WEIGHTS_DATASET,'as input next session to resume')
